In [2]:
import os
import numpy as np

os.chdir(r"C:\Users\Mohan2\Downloads\casp12.tar\casp12")

Loop through all the training sequences, count all the transitions and emissions. Disregard any masked residues

In [3]:
# 0 = Helix, 1 = sheet, 2 = loop

#emission matrix
EM = {0: [0]*21,
      1: [0]*21,
      2: [0]*21}

EM[0][20] = 0.333
EM[1][20] = 0.333
EM[2][20] = 0.333

#transition matrix
TM = {0: [0]*3,
      1: [0]*3,
      2: [0]*3}

#distribution for first and last residues
Ends = [0]*3

a = np.load('np_data.npz')
print(len(a.files))

for file in a.files:
    primary = a[file][0] 
    mask    = a[file][1] 
    DSSP    = a[file][2] 
    
    seqlen = len(primary)
    
    #first residue
    if mask[0]: #this is a non-masked residue. 0 is masked, 1 non-mask
        Ends[DSSP[0]] += 1
        EM[DSSP[0]][primary[0]] += 1
    
    #loop
    for i in range(1, seqlen):
        if mask[i]: #if current residue is unmasked, then count emission
            EM[DSSP[i]][primary[i]] += 1
            
            if mask[i-1]: #if prev residue is also unmasked, then count the transition
                TM[DSSP[i-1]][DSSP[i]] += 1
                
    #last residue
    if mask[seqlen-1]: 
        Ends[DSSP[0]] += 1
  


21008


Convert the counts into frequencies, and store as numpy arrays

In [4]:

# TM = np.load('HMM_params.npz')['TM'] #transition matrix values (log values), shape 3x3
#transmission matrix should have shape 3x3. residue i is the row, residue i+1 is the column
#in the dictionary the dict key is residue i, the list is residue i+1
np_TM = np.array([[x/sum(TM[0]) for x in TM[0]], [x/sum(TM[1]) for x in TM[1]], [x/sum(TM[2]) for x in TM[2]]])

#Emission matrix should have shape 21x3
np_EM = np.array([[x/sum(EM[0]) for x in EM[0]], [x/sum(EM[1]) for x in EM[1]], [x/sum(EM[2]) for x in EM[2]]]).T

np.savez('HMM_params.npz', np_TM = np_TM, np_EM = np_EM)